# EPIC v4 Generalized Pipeline\n\nEPIC v4 extends EPIC v3 by making the pipeline query-configurable.\n\nKey changes:\n- query-specific assumptions now live in `QUERY_CONFIGS`\n- directional scaffolding is query-relative: `option_a`, `option_b`, `hybrid`, `unclear`\n- artifacts are namespaced by query and retrieval setting\n- the same notebook can support the full planned multi-query study\n

## Setup

In [ ]:
# Optional on a fresh machine:
# %pip install openai pandas numpy scikit-learn tqdm openpyxl sentence-transformers python-dotenv nbformat rank-bm25

import json
import math
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi


def detect_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'epistemic-rag').exists():
            return candidate
        if (candidate / 'data' / 'prepared_corpora').exists() and (candidate / 'runs').exists():
            return candidate
    return start


ROOT = detect_root(Path.cwd())
BUNDLE_MODE = (ROOT / 'data' / 'prepared_corpora').exists() and (ROOT / 'runs').exists()
CORPUS_DIR = ROOT / 'data' / 'prepared_corpora' if BUNDLE_MODE else ROOT / 'epistemic-rag' / 'prepared_team_corpus'
DATA_DIR = ROOT / 'runs' if BUNDLE_MODE else ROOT / 'epistemic-rag' / 'data_v4'
DATA_DIR.mkdir(parents=True, exist_ok=True)

# CHANGE FROM EPIC v3:
# EPIC v4 introduces a query-config layer so the pipeline can generalize beyond q1
# without hardcoding student-loan-specific direction labels or corpus paths.
QUERY_CONFIGS = {
    'q1': {
        'query': 'Should I pay off student loans or start investing?',
        'corpus_filename': 'q1_corpus_prepared.xlsx',
        'option_a_label': 'pay off student loans',
        'option_b_label': 'start investing',
        'reason_types': [
            'interest_rate', 'tax', 'risk_tolerance', 'forgiveness',
            'psychology', 'liquidity', 'retirement_match', 'investment_return'
        ],
        'top_k_default': 10,
        'max_final_clusters': 6,
        'summary_label_examples': [
            'High-Interest Debt First',
            'Debt Freedom and Peace of Mind',
            'Balanced Approach: Debt and Investment',
            'Invest Over Low-Interest Loans',
            'Tax-Advantaged Investing First',
        ],
    },
    'q2': {
        'query': 'Should I contribute to a Roth IRA or a traditional 401k?',
        'corpus_filename': 'q2_corpus_prepared.xlsx',
        'option_a_label': 'contribute to a Roth IRA',
        'option_b_label': 'contribute to a traditional 401k',
        'reason_types': [
            'tax', 'employer_match', 'income_level', 'time_horizon',
            'withdrawal_rules', 'liquidity', 'retirement_rules'
        ],
        'top_k_default': 50,
        'max_final_clusters': 6,
        'summary_label_examples': [
            'Roth for Future Tax Flexibility',
            'Traditional 401k for Tax Deferral',
            'Use Both Strategically',
        ],
        'extraction_guidance': (
            'For this query, keep the account types straight. Treat Roth IRA, Roth 401k, '
            'traditional IRA, and traditional 401k as distinct account types. Use option_a only '
            'when the argument explicitly supports a Roth IRA or clearly recommends Roth-style '
            'after-tax retirement saving in a way that directly maps to choosing Roth IRA over a '
            'traditional 401k. Use option_b only when the argument explicitly supports a traditional '
            '401k or clearly recommends employer-plan pretax deferral over Roth IRA. If the source is '
            'mainly about Roth vs traditional tax treatment in general, IRA vs 401k generally, or Roth 401k '
            'vs traditional 401k without a clear mapping to the exact query, prefer unclear rather than forcing '
            'it into option_a or option_b.'
        ),
    },
    'q3': {
        'query': 'Should I build an emergency fund or pay down credit card debt first?',
        'corpus_filename': 'q3_corpus_prepared.xlsx',
        'option_a_label': 'build an emergency fund',
        'option_b_label': 'pay down credit card debt first',
        'reason_types': [
            'liquidity', 'interest_rate', 'risk_tolerance', 'job_stability',
            'cash_flow', 'psychology'
        ],
        'top_k_default': 10,
        'max_final_clusters': 6,
        'summary_label_examples': [
            'Emergency Buffer First',
            'High-Interest Debt First',
            'Build a Small Buffer Then Attack Debt',
        ],
    },
    'q4': {
        'query': 'Is buying a home better than renting given current interest rates?',
        'corpus_filename': 'q4_corpus_prepared.xlsx',
        'option_a_label': 'buy a home',
        'option_b_label': 'rent',
        'reason_types': [
            'interest_rate', 'housing_market', 'mobility', 'time_horizon',
            'maintenance_cost', 'cash_flow', 'equity_building', 'uncertainty'
        ],
        'top_k_default': 10,
        'max_final_clusters': 6,
        'summary_label_examples': [
            'Buy for Long-Term Stability',
            'Rent for Flexibility',
            'Depends on Holding Period',
        ],
    },
    'q5': {
        'query': 'Should I invest in index funds or pay off my mortgage early?',
        'corpus_filename': 'q5_corpus_prepared.xlsx',
        'option_a_label': 'invest in index funds',
        'option_b_label': 'pay off my mortgage early',
        'reason_types': [
            'interest_rate', 'investment_return', 'tax', 'risk_tolerance',
            'psychology', 'liquidity', 'time_horizon'
        ],
        'top_k_default': 10,
        'max_final_clusters': 6,
        'summary_label_examples': [
            'Invest Over Low-Rate Mortgage',
            'Mortgage Freedom First',
            'Do Both Strategically',
        ],
    },
}

ACTIVE_QUERY_ID = 'q2'
ACTIVE_CONFIG = QUERY_CONFIGS[ACTIVE_QUERY_ID]
QUERY_ID = ACTIVE_QUERY_ID
QUERY_TEXT = ACTIVE_CONFIG['query']
OPTION_A_LABEL = ACTIVE_CONFIG['option_a_label']
OPTION_B_LABEL = ACTIVE_CONFIG['option_b_label']
OPTION_A_SHORT = OPTION_A_LABEL.title()
OPTION_B_SHORT = OPTION_B_LABEL.title()
REASON_TYPES = ACTIVE_CONFIG['reason_types']
TOP_K = ACTIVE_CONFIG.get('top_k_default', 10)
MAX_FINAL_CLUSTERS = ACTIVE_CONFIG.get('max_final_clusters', 6)
SUMMARY_LABEL_EXAMPLES = ACTIVE_CONFIG.get('summary_label_examples', [])
EXTRACTION_GUIDANCE = ACTIVE_CONFIG.get('extraction_guidance', '')
RUN_NAME = f'{QUERY_ID}_topk{TOP_K}'
RUN_DIR = DATA_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = CORPUS_DIR / ACTIVE_CONFIG['corpus_filename']
if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f'Corpus file not found for {QUERY_ID}: {CORPUS_PATH}. '
        'Update QUERY_CONFIGS or add the corpus file before running this query.'
    )

# CHANGE FROM EPIC v3:
# Artifacts are namespaced by query and top_k so multi-query sweeps do not overwrite each other.
CLEAN_PATH = DATA_DIR / f'{QUERY_ID}_clean_v4.json'
RETRIEVED_DOCS_PATH = RUN_DIR / 'retrieved_docs.json'
STRUCTURED_PATH = RUN_DIR / 'structured_arguments.json'
SCORED_PATH = RUN_DIR / 'structured_scored.json'
CLAIM_EMBED_PATH = RUN_DIR / 'claim_embeddings.npy'
TEXT_EMBED_PATH = RUN_DIR / 'support_embeddings.npy'
INDEXED_PATH = RUN_DIR / 'arguments_indexed.json'
CLUSTERS_PATH = RUN_DIR / 'perspective_clusters.json'
SUMMARIES_PATH = RUN_DIR / 'perspective_summaries.json'
FINAL_OUTPUT_PATH = RUN_DIR / 'final_output.json'
EVAL_PATH = RUN_DIR / 'evaluation.json'
TIMINGS_PATH = RUN_DIR / 'step_times.json'
LOG_PATH = RUN_DIR / 'logs.json'

for env_candidate in [ROOT / '.env', Path.cwd() / '.env', ROOT.parent / '.env']:
    if env_candidate.exists():
        load_dotenv(env_candidate)
        break
API_KEY = os.getenv('OPENAI_API_KEY') or os.environ.get('OPENAI_API_KEY')
if not API_KEY:
    raise ValueError('OPENAI_API_KEY not found in .env or environment variables.')

client = OpenAI(api_key=API_KEY)

LINE_DROP_PATTERNS = ['Related Articles', 'Article Sources', 'Partner Links', '*** PASTE']
INLINE_REMOVE_PATTERNS = ['Get personalized', 'Ask anything to get started...', 'ASK', 'Read more', 'Archive', 'Advertisement']
SENTENCE_DROP_PATTERNS = [
    'written by', 'co-written by', 'edited by', 'fact checked', 'updated ', 'min read',
    'read time', 'share article via', 'share:', 'photo by', 'getty images', 'paid placement',
    'terms apply', 'learn more', 'our top picks', 'select independently determines',
    'click to what you’re looking for', "click to what you're looking for", 'close key takeaways'
]

def save_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def load_json(path, default=None):
    if not path.exists():
        return default
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def record_log(key, message):
    logs = load_json(LOG_PATH, default={})
    logs.setdefault(key, []).append(message)
    save_json(LOG_PATH, logs)

def start_timer():
    return time.time()

def stop_timer(step_name, started_at):
    elapsed = round(time.time() - started_at, 2)
    timings = load_json(TIMINGS_PATH, default={})
    timings[step_name] = elapsed
    save_json(TIMINGS_PATH, timings)
    return elapsed

def word_count(text):
    return len(re.findall(r'\b\w+\b', str(text)))

def sentence_count(text):
    pieces = re.split(r'(?<=[.!?])\s+', str(text).strip())
    return len([piece for piece in pieces if piece.strip()])

def is_url_only(line):
    return bool(re.fullmatch(r'https?://\S+', line.strip()))

def should_drop_sentence(sentence):
    lowered = sentence.lower().strip()
    if not lowered:
        return True
    if any(pattern in lowered for pattern in SENTENCE_DROP_PATTERNS) and word_count(sentence) <= 35:
        return True
    if re.fullmatch(r'[A-Z][A-Za-z ]{0,40}\|\s+[A-Za-z]+\s+\d{1,2},\s+\d{4}', sentence):
        return True
    if re.fullmatch(r'.{0,120}u/[A-Za-z0-9_\-]+:?$', sentence):
        return True
    return False

def clean_raw_text(text):
    text = str(text)
    text = re.sub(r'\[(\d+)\]', '', text)
    text = re.sub(r'Get personalized,.*?trusted expertise\.', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'Share Article via [A-Za-z ]+', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'Learn More Terms Apply Paid Placement', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\b[A-Z][a-z]+\s+\d{1,2},\s+\d{4}\b', ' ', text)
    for pattern in INLINE_REMOVE_PATTERNS:
        text = re.sub(re.escape(pattern), ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        lines = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9\[])', text)

    kept = []
    for line in lines:
        line = re.sub(r'\s+', ' ', line).strip()
        if not line or is_url_only(line):
            continue
        if any(pattern.lower() == line.lower() for pattern in LINE_DROP_PATTERNS):
            continue
        if word_count(line) < 6:
            continue
        if should_drop_sentence(line):
            continue
        kept.append(line)
    cleaned = '\n'.join(kept)
    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned)
    return cleaned.strip()

def softmax(values):
    arr = np.array(values, dtype=float)
    arr = arr - np.max(arr)
    exp = np.exp(arr)
    return exp / exp.sum()

def extract_entailment_scores(raw_scores):
    arr = np.array(raw_scores, dtype=float)
    if arr.ndim == 1:
        return arr
    if arr.ndim == 2 and arr.shape[1] >= 3:
        return arr[:, 2]
    raise ValueError(f'Unexpected CrossEncoder score shape: {arr.shape}')

def call_openai_json(model, system_prompt, user_prompt, temperature=0.0, retry_wait=5):
    for attempt in range(2):
        try:
            response = client.chat.completions.create(
                model=model,
                temperature=temperature,
                response_format={'type': 'json_object'},
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': user_prompt},
                ],
            )
            return json.loads(response.choices[0].message.content)
        except Exception as exc:
            if attempt == 0:
                print(f'OpenAI call failed, retrying once in {retry_wait}s: {exc}')
                time.sleep(retry_wait)
            else:
                raise

def posture_from_score(score):
    if score > 0.6:
        return 'confident'
    if score >= 0.3:
        return 'conditional'
    return 'informational'

def confidence_label_from_postures(posture_distribution):
    if posture_distribution.get('confident', 0) >= max(posture_distribution.get('conditional', 0), posture_distribution.get('informational', 0)):
        return 'Confident'
    if posture_distribution.get('conditional', 0) >= posture_distribution.get('informational', 0):
        return 'Conditional'
    return 'Informational'

def mean_pairwise_cosine(matrix):
    if len(matrix) < 2:
        return 1.0
    sims = cosine_similarity(matrix)
    tri = sims[np.triu_indices(len(matrix), k=1)]
    return float(np.mean(tri)) if len(tri) else 1.0

print('EPIC v4 setup complete.')
print(f'Layout mode: {"submission_bundle" if BUNDLE_MODE else "project"}')
print(f'Working directory: {ROOT}')
print(f'Corpus directory: {CORPUS_DIR}')
print(f'Output directory: {DATA_DIR}')
print(f'Run directory: {RUN_DIR}')
print(f'Run name: {RUN_NAME}')
print(f'Active query: {QUERY_ID}')
print(f'Query text: {QUERY_TEXT}')
print(f'Option A: {OPTION_A_LABEL}')
print(f'Option B: {OPTION_B_LABEL}')


## STEP 1: Load and clean corpus

This step stays close to EPIC v2. The main new change comes immediately after this step: BM25 retrieval over the cleaned corpus.

In [ ]:
step_start = start_timer()

df = pd.read_excel(CORPUS_PATH)
required_cols = ['id', 'query_id', 'url', 'source_type', 'source_name', 'raw_text']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

df['raw_text'] = df['raw_text'].fillna('').astype(str)
df['word_count'] = df['raw_text'].apply(word_count)
df = df[df['word_count'] >= 50].copy()
df['query'] = QUERY_TEXT
df['raw_text'] = df['raw_text'].apply(clean_raw_text)
clean_docs = df.to_dict(orient='records')
save_json(CLEAN_PATH, clean_docs)

print(f'doc count: {len(clean_docs)}')
print('source_type breakdown:', dict(df['source_type'].value_counts().sort_index()))
print(f"avg original word count: {float(df['word_count'].mean()):.2f}")

elapsed = stop_timer('step_1_load_and_clean_v2', step_start)
print(f'Step 1 time: {elapsed:.2f}s')

## STEP 2: BM25 candidate retrieval

### CHANGE FROM EPIC v2
EPIC v2 decomposed the entire cleaned corpus directly. EPIC v3 first retrieves a **candidate pool** with BM25, then runs structured argument decomposition only on that pool.

Why:
- makes the pipeline much closer to real RAG
- lets us study discovery within a retrieved set rather than a hand-curated slice
- sets up later generalization to larger multi-query corpora

In [ ]:
step_start = start_timer()

docs = load_json(CLEAN_PATH, default=[])
if not docs:
    raise ValueError('No cleaned docs found. Run Step 1 first.')

# CHANGE FROM EPIC v2:
# Retrieval happens before decomposition. We score each cleaned document with BM25
# using the raw query text, then keep only the top candidate pool.

def bm25_tokenize(text):
    return re.findall(r'\b\w+\b', str(text).lower())

corpus_tokens = [bm25_tokenize(doc['raw_text']) for doc in docs]
query_tokens = bm25_tokenize(QUERY_TEXT)
bm25 = BM25Okapi(corpus_tokens)
scores = bm25.get_scores(query_tokens)

scored_docs = []
for rank, (doc, score) in enumerate(sorted(zip(docs, scores), key=lambda x: x[1], reverse=True), start=1):
    item = dict(doc)
    item['bm25_score'] = float(score)
    item['retrieval_rank'] = int(rank)
    scored_docs.append(item)

retrieved_docs = scored_docs[: min(TOP_K, len(scored_docs))]
save_json(RETRIEVED_DOCS_PATH, retrieved_docs)

print(f'retrieved candidate docs: {len(retrieved_docs)}')
print('candidate source_type breakdown:', dict(sorted(Counter(doc['source_type'] for doc in retrieved_docs).items())))
if len(retrieved_docs) == len(docs):
    print('note: BM25 did not prune this corpus because top_k exceeds available docs; this is expected on the small q1 prototype corpus.')
print('top 5 BM25 docs:')
for doc in retrieved_docs[:5]:
    print('-', doc['id'], '|', doc['source_name'], '|', doc['source_type'], '| rank =', doc['retrieval_rank'], '| score =', round(doc['bm25_score'], 3))

elapsed = stop_timer('step_2_bm25_retrieval_v3', step_start)
print(f'Step 2 time: {elapsed:.2f}s')


## STEP 3: Structured argument decomposition

### CHANGE FROM EPIC v2
The original EPIC notebook extracted only free-text arguments. EPIC v2 extracts a **main claim** plus **conditions** and **caveats** separately.

Why:
- `main_claim` should drive viewpoint discovery.
- `conditions` should not dominate clustering.
- `caveats` should enrich summaries rather than create fake perspectives.

In [ ]:
step_start = start_timer()

STRUCTURED_FORCE_REBUILD = True
docs = load_json(RETRIEVED_DOCS_PATH, default=[])
existing = [] if STRUCTURED_FORCE_REBUILD else load_json(STRUCTURED_PATH, default=[])
processed_doc_ids = {item['source_doc_id'] for item in existing}
structured_arguments = list(existing)
if STRUCTURED_FORCE_REBUILD and STRUCTURED_PATH.exists():
    STRUCTURED_PATH.unlink()

reason_type_prompt = ', '.join(REASON_TYPES)
direction_instruction = (
    f'use "option_a" when the argument mainly favors {OPTION_A_LABEL}; '
    f'use "option_b" when it mainly favors {OPTION_B_LABEL}; '
    'use "hybrid" when it advocates doing both or splitting the difference; '
    'use "unclear" for meta-advice or weakly directional content.'
)
extra_guidance_block = f"\n- Additional query-specific guidance: {EXTRACTION_GUIDANCE}" if EXTRACTION_GUIDANCE else ''

system_prompt = (
    'You extract structured financial arguments from documents. Use only the text provided. '
    'Do not invent claims, conditions, or caveats. Return only valid JSON.'
)

for doc in tqdm(docs, desc='Structured extraction'):
    if doc['id'] in processed_doc_ids:
        continue

    user_prompt = f'''Query: {QUERY_TEXT}
Option A: {OPTION_A_LABEL}
Option B: {OPTION_B_LABEL}
Source type: {doc['source_type']}
Source name: {doc['source_name']}

Document text:
{doc['raw_text']}

Extract all distinct arguments relevant to the query.

IMPORTANT:
- Separate the MAIN CLAIM from CONDITIONS and CAVEATS.
- The main claim should be the core recommendation or position, stated in 1-2 sentences.
- Conditions are circumstances under which the claim holds.
- Caveats are warnings, uncertainties, exceptions, or limitations.
- reason_types should come from this vocabulary where possible: {reason_type_prompt}.
- supporting_text should stay readable and preserve the source voice in 2-5 sentences.
- Do not let condition language replace the main claim.
- For coarse_direction, {direction_instruction}
{extra_guidance_block}

If this is a community source, treat each distinct commenter position as a separate argument.

Return ONLY valid JSON:
{{
  "arguments": [
    {{
      "index": 0,
      "main_claim": "...",
      "supporting_text": "...",
      "conditions": ["...", "..."],
      "caveats": ["...", "..."],
      "reason_types": ["interest_rate", "tax"],
      "coarse_direction": "option_a" or "option_b" or "hybrid" or "unclear"
    }}
  ]
}}

If no clear arguments exist, return {{"arguments": []}}.'''

    result = call_openai_json('gpt-4o-mini', system_prompt, user_prompt, temperature=0.0)
    arguments = result.get('arguments', [])
    if not arguments:
        record_log('empty_structured_extractions', f'No arguments extracted for {doc["id"]}')

    for i, arg in enumerate(arguments):
        idx = arg.get('index', i)
        main_claim = str(arg.get('main_claim', '')).strip()
        supporting_text = str(arg.get('supporting_text', '')).strip()
        if not main_claim or not supporting_text:
            continue
        if word_count(main_claim) < 8 or sentence_count(supporting_text) < 2:
            record_log('dropped_understructured_arguments', f'Dropped weak structured argument for {doc["id"]}_a{idx}')
            continue
        item = {
            'id': f"{doc['id']}_a{idx}",
            'query_id': QUERY_ID,
            'query': QUERY_TEXT,
            'source_doc_id': doc['id'],
            'source_type': doc['source_type'],
            'source_name': doc['source_name'],
            'url': doc['url'],
            'bm25_score': float(doc.get('bm25_score', 0.0)),
            'retrieval_rank': int(doc.get('retrieval_rank', 0)),
            'main_claim': main_claim,
            'supporting_text': supporting_text,
            'conditions': [str(x).strip() for x in arg.get('conditions', []) if str(x).strip()],
            'caveats': [str(x).strip() for x in arg.get('caveats', []) if str(x).strip()],
            'reason_types': [str(x).strip() for x in arg.get('reason_types', []) if str(x).strip()],
            'coarse_direction': str(arg.get('coarse_direction', 'unclear')).strip().lower(),
            'main_claim_word_count': word_count(main_claim),
            'supporting_word_count': word_count(supporting_text),
        }
        structured_arguments.append(item)

    save_json(STRUCTURED_PATH, structured_arguments)
    processed_doc_ids.add(doc['id'])
    time.sleep(0.5)

print(f'total structured arguments: {len(structured_arguments)}')
print('arguments by source_type:', dict(sorted(Counter(item['source_type'] for item in structured_arguments).items())))
print('coarse direction breakdown:', dict(sorted(Counter(item['coarse_direction'] for item in structured_arguments).items())))
for sample in structured_arguments[:3]:
    print('-' * 100)
    print(sample['id'], '| claim =', sample['main_claim'])
    print('conditions:', sample['conditions'])
    print('caveats:', sample['caveats'])

elapsed = stop_timer('step_3_structured_extraction_v4', step_start)
print(f'Step 3 time: {elapsed:.2f}s')


## STEP 4: Epistemic posture scoring

### CHANGE FROM EPIC v2
We still score epistemic posture, but only after BM25 retrieval has limited the candidate pool. This makes posture estimation part of the post-retrieval organization stage rather than a full-corpus preprocessing step.

In [ ]:
from sentence_transformers import CrossEncoder

step_start = start_timer()
arguments = load_json(STRUCTURED_PATH, default=[])
if not arguments:
    raise ValueError('No structured arguments found. Run Step 2 first.')

existing = load_json(SCORED_PATH, default=[])
existing_map = {item['id']: item for item in existing}
nli_model = CrossEncoder('cross-encoder/nli-deberta-v3-base')

h1 = 'This text makes a clear confident financial recommendation'
h2 = 'This text says the answer depends on individual circumstances'
h3 = 'This text is primarily explanatory or informational rather than a direct recommendation'

scored = []
for item in tqdm(arguments, desc='Scoring posture'):
    if item['id'] in existing_map:
        scored.append(existing_map[item['id']])
        continue

    score_text = f"Main claim: {item['main_claim']}\nSupport: {item['supporting_text']}"
    raw_scores = nli_model.predict([[score_text, h1], [score_text, h2], [score_text, h3]])
    scores = extract_entailment_scores(raw_scores)
    probs = softmax(scores)
    out = dict(item)
    out.update({
        'confidence_score': float(probs[0]),
        'posture': posture_from_score(float(probs[0])),
        'h1_prob': float(probs[0]),
        'h2_prob': float(probs[1]),
        'h3_prob': float(probs[2]),
    })
    scored.append(out)
    save_json(SCORED_PATH, scored)

save_json(SCORED_PATH, scored)
print('posture distribution:', dict(sorted(Counter(item['posture'] for item in scored).items())))

elapsed = stop_timer('step_4_posture_scoring_v3', step_start)
print(f'Step 3 time: {elapsed:.2f}s')

## STEP 5: Claim-centric embeddings

### CHANGE FROM EPIC v2
The original pipeline embedded full argument text only. EPIC v2 separately embeds:
- `main_claim` embeddings for viewpoint discovery
- `supporting_text` embeddings for later summarization and sanity checks

This keeps discovery focused on the core recommendation rather than on the surrounding caveat language.

In [ ]:
from sentence_transformers import SentenceTransformer

step_start = start_timer()
arguments = load_json(SCORED_PATH, default=[])
if not arguments:
    raise ValueError('No scored arguments found. Run Step 3 first.')

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
claim_texts = [item['main_claim'] for item in arguments]
support_texts = [item['supporting_text'] for item in arguments]
claim_embeddings = embed_model.encode(claim_texts, show_progress_bar=True, convert_to_numpy=True)
support_embeddings = embed_model.encode(support_texts, show_progress_bar=True, convert_to_numpy=True)

np.save(CLAIM_EMBED_PATH, claim_embeddings)
np.save(TEXT_EMBED_PATH, support_embeddings)
save_json(INDEXED_PATH, arguments)

print('claim embedding shape:', claim_embeddings.shape)
print('support embedding shape:', support_embeddings.shape)

elapsed = stop_timer('step_5_claim_embeddings_v3', step_start)
print(f'Step 4 time: {elapsed:.2f}s')

## STEP 6: EPIC v4 perspective discovery

In [ ]:
step_start = start_timer()
arguments = load_json(INDEXED_PATH, default=[])
claim_embeddings = np.load(CLAIM_EMBED_PATH)
support_embeddings = np.load(TEXT_EMBED_PATH)
if len(arguments) != len(claim_embeddings):
    raise ValueError('Argument count and claim embedding count do not match.')
arg_id_to_idx = {item['id']: idx for idx, item in enumerate(arguments)}

# CHANGE FROM EPIC v3:
# Direction buckets are now query-relative. This preserves the general EPIC logic while
# removing student-loan-specific supervision from the clustering layer.
distance_threshold = 0.34
max_final_clusters = MAX_FINAL_CLUSTERS
min_cluster_size = 3
primary_directions = ['option_a', 'option_b', 'hybrid']


def build_cluster(cluster_id, idxs):
    items = [dict(arguments[i]) for i in idxs]
    centroid = np.mean(claim_embeddings[idxs], axis=0)
    condition_counts = Counter()
    caveat_counts = Counter()
    reason_counts = Counter()
    for item in items:
        condition_counts.update(item['conditions'])
        caveat_counts.update(item['caveats'])
        reason_counts.update(item['reason_types'])
    return {
        'cluster_id': cluster_id,
        'passages': items,
        'size': len(items),
        'mean_confidence': float(np.mean([item['confidence_score'] for item in items])),
        'posture_distribution': dict(sorted(Counter(item['posture'] for item in items).items())),
        'source_types': sorted({item['source_type'] for item in items}),
        'source_names': sorted({item['source_name'] for item in items}),
        'coarse_direction_distribution': dict(sorted(Counter(item['coarse_direction'] for item in items).items())),
        'top_conditions': [x for x, _ in condition_counts.most_common(8)],
        'top_caveats': [x for x, _ in caveat_counts.most_common(8)],
        'top_reason_types': [x for x, _ in reason_counts.most_common(8)],
        'centroid': centroid.tolist(),
        'purity_hint': mean_pairwise_cosine(claim_embeddings[idxs]),
    }


def rerender_cluster(cluster):
    idxs = [arg_id_to_idx[item['id']] for item in cluster['passages'] if item['id'] in arg_id_to_idx]
    return build_cluster(cluster['cluster_id'], idxs)


def cluster_within_direction(direction, idxs):
    if not idxs:
        return []
    if len(idxs) == 1:
        return [build_cluster(f'{direction}_0', idxs)]
    subset = claim_embeddings[idxs]
    clusterer = AgglomerativeClustering(
        n_clusters=None,
        metric='cosine',
        linkage='average',
        distance_threshold=distance_threshold,
    )
    labels = clusterer.fit_predict(subset)
    local_groups = {}
    for local_pos, label in enumerate(labels):
        local_groups.setdefault(int(label), []).append(idxs[local_pos])
    return [build_cluster(f'{direction}_{label}', group_idxs) for label, group_idxs in local_groups.items()]


def nearest_non_unclear_cluster(item):
    item_idx = arg_id_to_idx[item['id']]
    item_emb = claim_embeddings[item_idx]
    candidates = [c for c in cluster_items if 'unclear' not in c['coarse_direction_distribution'] or len(c['coarse_direction_distribution']) > 1]
    if not candidates:
        candidates = cluster_items
    return max(candidates, key=lambda c: float(cosine_similarity([item_emb], [c['centroid']])[0][0]))


cluster_items = []
for direction in primary_directions:
    direction_idxs = [idx for idx, item in enumerate(arguments) if item['coarse_direction'] == direction]
    cluster_items.extend(cluster_within_direction(direction, direction_idxs))

unclear_items = [item for item in arguments if item['coarse_direction'] == 'unclear']
for item in unclear_items:
    if not cluster_items:
        cluster_items.append(build_cluster('unclear_0', [arg_id_to_idx[item['id']]]))
        continue
    target = nearest_non_unclear_cluster(item)
    target['passages'].append(dict(item))
    refreshed = rerender_cluster(target)
    for idx, cluster in enumerate(cluster_items):
        if cluster['cluster_id'] == target['cluster_id']:
            cluster_items[idx] = refreshed
            break


def dominant_direction(cluster):
    if not cluster['coarse_direction_distribution']:
        return 'unclear'
    return max(cluster['coarse_direction_distribution'].items(), key=lambda kv: kv[1])[0]


while True:
    small = [c for c in cluster_items if c['size'] < min_cluster_size]
    if not small:
        break
    cluster = min(small, key=lambda c: c['size'])
    same_direction = [c for c in cluster_items if c is not cluster and dominant_direction(c) == dominant_direction(cluster)]
    others = [c for c in cluster_items if c is not cluster]
    pool = same_direction if same_direction else others
    if not pool:
        break
    target = max(pool, key=lambda c: float(cosine_similarity([cluster['centroid']], [c['centroid']])[0][0]))
    target['passages'].extend(cluster['passages'])
    refreshed = rerender_cluster(target)
    new_clusters = []
    for c in cluster_items:
        if c is cluster:
            continue
        if c is target:
            new_clusters.append(refreshed)
        else:
            new_clusters.append(c)
    cluster_items = new_clusters

cluster_items = sorted(cluster_items, key=lambda c: (c['mean_confidence'], c['size']), reverse=True)
selected = []
covered_directions = set()
for cluster in cluster_items:
    direction = dominant_direction(cluster)
    if direction not in covered_directions:
        selected.append(cluster)
        covered_directions.add(direction)
for cluster in cluster_items:
    if len(selected) >= max_final_clusters:
        break
    if cluster not in selected:
        selected.append(cluster)
cluster_items = selected[:max_final_clusters]

cluster_items = sorted(cluster_items, key=lambda c: (c['mean_confidence'], c['size']), reverse=True)
for idx, cluster in enumerate(cluster_items):
    cluster['cluster_id'] = f'c{idx}'

save_json(CLUSTERS_PATH, cluster_items)

for cluster in cluster_items:
    print('\n' + '=' * 100)
    print(cluster['cluster_id'], '| size =', cluster['size'], '| mean_confidence =', round(cluster['mean_confidence'], 3))
    print('source_types =', cluster['source_types'])
    print('posture_distribution =', cluster['posture_distribution'])
    print('coarse_direction_distribution =', cluster['coarse_direction_distribution'])
    print('top_reason_types =', cluster['top_reason_types'][:5])
    print('top_conditions =', cluster['top_conditions'][:3])
    print('top_caveats =', cluster['top_caveats'][:3])
    for sample in cluster['passages'][:3]:
        print('-', sample['main_claim'][:120])

elapsed = stop_timer('step_6_epic_v4_discovery', step_start)
print(f'Step 6 time: {elapsed:.2f}s')


## STEP 7: Intrinsic evaluation

In [ ]:
step_start = start_timer()
arguments = load_json(INDEXED_PATH, default=[])
clusters = load_json(CLUSTERS_PATH, default=[])
claim_embeddings = np.load(CLAIM_EMBED_PATH)
arg_index = {item['id']: idx for idx, item in enumerate(arguments)}

# CHANGE FROM EPIC v3:
# Evaluation is now query-relative too. The ablations stay the same, but direction coverage
# is measured over option_a / option_b / hybrid instead of a student-loan-specific schema.


def cluster_matrix(cluster):
    idxs = [arg_index[item['id']] for item in cluster['passages'] if item['id'] in arg_index]
    return claim_embeddings[idxs]


def normalized_entropy(counter_dict, allowed_labels):
    counts = np.array([counter_dict.get(label, 0) for label in allowed_labels], dtype=float)
    total = counts.sum()
    if total == 0:
        return 0.0
    probs = counts / total
    probs = probs[probs > 0]
    if len(probs) <= 1:
        return 0.0
    entropy = -np.sum(probs * np.log(probs))
    return float(entropy / np.log(len(allowed_labels)))


def evaluate_cluster_set(cluster_set):
    per_cluster_purity = []
    centroids = []
    represented_postures = set()
    represented_directions = set()
    posture_counter = Counter()
    direction_counter = Counter()
    for cluster in cluster_set:
        matrix = cluster_matrix(cluster)
        if len(matrix) == 0:
            continue
        purity = mean_pairwise_cosine(matrix)
        per_cluster_purity.append(purity)
        centroids.append(np.mean(matrix, axis=0))
        cluster_postures = [item['posture'] for item in cluster['passages']]
        cluster_directions = [item['coarse_direction'] for item in cluster['passages']]
        represented_postures.update(cluster_postures)
        represented_directions.update(cluster_directions)
        posture_counter.update(cluster_postures)
        direction_counter.update([d for d in cluster_directions if d != 'unclear'])

    purity_mean = float(np.mean(per_cluster_purity)) if per_cluster_purity else 0.0
    if len(centroids) >= 2:
        centroid_matrix = np.array(centroids)
        sims = cosine_similarity(centroid_matrix)
        tri = sims[np.triu_indices(len(centroid_matrix), k=1)]
        inter_dist = float(np.mean(1 - tri)) if len(tri) else 0.0
    else:
        inter_dist = 0.0

    ecs = 0.0
    for idx, cluster in enumerate(cluster_set[:len(per_cluster_purity)]):
        purity_c = per_cluster_purity[idx]
        if len(centroids) > 1:
            others = [centroids[j] for j in range(len(centroids)) if j != idx]
            sim_to_others = float(np.mean(cosine_similarity([centroids[idx]], others)[0]))
        else:
            sim_to_others = 0.0
        ecs += purity_c * math.log(max(len(cluster_set), 1) / (sim_to_others + 1e-9))

    posture_coverage = len(represented_postures) / 3.0
    direction_coverage = len(represented_directions - {'unclear'}) / 3.0
    posture_balance = normalized_entropy(posture_counter, ['confident', 'conditional', 'informational'])
    direction_balance = normalized_entropy(direction_counter, ['option_a', 'option_b', 'hybrid'])
    coverage_adjusted_ecs = float(ecs) * (0.5 * direction_coverage + 0.5 * posture_coverage)
    balance_adjusted_ecs = float(ecs) * (0.5 * direction_balance + 0.5 * posture_balance)

    return {
        'num_clusters': len(cluster_set),
        'purity_per_cluster': per_cluster_purity,
        'purity_mean': purity_mean,
        'inter_cluster_distance': inter_dist,
        'ecs': float(ecs),
        'posture_coverage': posture_coverage,
        'direction_coverage': direction_coverage,
        'posture_balance': posture_balance,
        'direction_balance': direction_balance,
        'coverage_adjusted_ecs': coverage_adjusted_ecs,
        'balance_adjusted_ecs': balance_adjusted_ecs,
    }


def build_cluster_from_indices(cluster_id, idxs):
    items = [dict(arguments[i]) for i in idxs]
    condition_counts = Counter()
    caveat_counts = Counter()
    reason_counts = Counter()
    for item in items:
        condition_counts.update(item['conditions'])
        caveat_counts.update(item['caveats'])
        reason_counts.update(item['reason_types'])
    return {
        'cluster_id': cluster_id,
        'passages': items,
        'size': len(items),
        'mean_confidence': float(np.mean([item['confidence_score'] for item in items])) if items else 0.0,
        'posture_distribution': dict(sorted(Counter(item['posture'] for item in items).items())),
        'source_types': sorted({item['source_type'] for item in items}),
        'source_names': sorted({item['source_name'] for item in items}),
        'coarse_direction_distribution': dict(sorted(Counter(item['coarse_direction'] for item in items).items())),
        'top_conditions': [x for x, _ in condition_counts.most_common(8)],
        'top_caveats': [x for x, _ in caveat_counts.most_common(8)],
        'top_reason_types': [x for x, _ in reason_counts.most_common(8)],
        'centroid': np.mean(claim_embeddings[idxs], axis=0).tolist() if idxs else [],
        'purity_hint': mean_pairwise_cosine(claim_embeddings[idxs]) if idxs else 0.0,
    }


def equal_groups(indices, num_groups, prefix):
    groups = []
    for i, split in enumerate(np.array_split(np.array(indices), num_groups)):
        split = split.tolist()
        if not split:
            continue
        groups.append(build_cluster_from_indices(f'{prefix}{i}', split))
    return groups


def semantic_only_global(indices, distance_threshold=0.34, max_clusters=6, min_cluster_size=3):
    if not indices:
        return []
    if len(indices) == 1:
        return [build_cluster_from_indices('semantic_0', indices)]
    subset = claim_embeddings[indices]
    clusterer = AgglomerativeClustering(
        n_clusters=None,
        metric='cosine',
        linkage='average',
        distance_threshold=distance_threshold,
    )
    labels = clusterer.fit_predict(subset)
    groups = {}
    for local_pos, label in enumerate(labels):
        groups.setdefault(int(label), []).append(indices[local_pos])
    cluster_items = [build_cluster_from_indices(f'semantic_{label}', idxs) for label, idxs in groups.items()]
    cluster_items = sorted(cluster_items, key=lambda c: (c['mean_confidence'], c['size']), reverse=True)
    while True:
        small = [c for c in cluster_items if c['size'] < min_cluster_size]
        if not small or len(cluster_items) <= 1:
            break
        cluster = min(small, key=lambda c: c['size'])
        others = [c for c in cluster_items if c is not cluster]
        target = max(others, key=lambda c: float(cosine_similarity([cluster['centroid']], [c['centroid']])[0][0]))
        merged_idxs = [arg_index[item['id']] for item in target['passages'] + cluster['passages']]
        merged = build_cluster_from_indices(target['cluster_id'], merged_idxs)
        new_clusters = []
        for c in cluster_items:
            if c is cluster:
                continue
            if c is target:
                new_clusters.append(merged)
            else:
                new_clusters.append(c)
        cluster_items = new_clusters
    return sorted(cluster_items, key=lambda c: (c['mean_confidence'], c['size']), reverse=True)[:max_clusters]


def direction_only_clusters():
    groups = []
    for direction in ['option_a', 'option_b', 'hybrid']:
        idxs = [idx for idx, item in enumerate(arguments) if item['coarse_direction'] == direction]
        if idxs:
            groups.append(build_cluster_from_indices(f'direction_{direction}', idxs))
    unclear = [idx for idx, item in enumerate(arguments) if item['coarse_direction'] == 'unclear']
    if unclear and groups:
        target = max(groups, key=lambda c: c['size'])
        merged_idxs = [arg_index[item['id']] for item in target['passages']] + unclear
        rebuilt = build_cluster_from_indices(target['cluster_id'], merged_idxs)
        groups = [rebuilt if c['cluster_id'] == target['cluster_id'] else c for c in groups]
    return groups


def direction_first_no_unclear_handling():
    groups = []
    for direction in ['option_a', 'option_b', 'hybrid', 'unclear']:
        idxs = [idx for idx, item in enumerate(arguments) if item['coarse_direction'] == direction]
        groups.extend(semantic_only_global(idxs, max_clusters=2) if idxs else [])
    return sorted(groups, key=lambda c: (c['mean_confidence'], c['size']), reverse=True)[:MAX_FINAL_CLUSTERS]


all_indices = list(range(len(arguments)))
evaluation = {}
evaluation['EPIC_v4'] = evaluate_cluster_set(clusters)
evaluation['Random'] = evaluate_cluster_set(equal_groups(np.random.default_rng(42).permutation(all_indices), len(clusters), 'random_'))
evaluation['RelevanceOrder'] = evaluate_cluster_set(equal_groups(all_indices, len(clusters), 'relevance_'))
evaluation['SemanticOnlyGlobal'] = evaluate_cluster_set(semantic_only_global(all_indices, max_clusters=len(clusters)))
evaluation['DirectionOnly'] = evaluate_cluster_set(direction_only_clusters())
evaluation['NoUnclearHandling'] = evaluate_cluster_set(direction_first_no_unclear_handling())

save_json(EVAL_PATH, evaluation)

print('Method             | Clusters | Purity | Inter-dist | ECS  | CovAdj | BalAdj | PostBal | DirBal')
for method_name in ['EPIC_v4', 'Random', 'RelevanceOrder', 'SemanticOnlyGlobal', 'DirectionOnly', 'NoUnclearHandling']:
    metrics = evaluation[method_name]
    print(
        f"{method_name:<18} | {metrics['num_clusters']:<8} | {metrics['purity_mean']:.2f}   | "
        f"{metrics['inter_cluster_distance']:.2f}       | {metrics['ecs']:.2f} | "
        f"{metrics['coverage_adjusted_ecs']:.2f}  | {metrics['balance_adjusted_ecs']:.2f}  | "
        f"{metrics['posture_balance']:.2f}    | {metrics['direction_balance']:.2f}"
    )

elapsed = stop_timer('step_7_intrinsic_eval_v4', step_start)
print(f'Step 7 time: {elapsed:.2f}s')


## STEP 8: Perspective summarization

In [ ]:
step_start = start_timer()
clusters = load_json(CLUSTERS_PATH, default=[])
SUMMARIES_FORCE_REBUILD = True
existing = [] if SUMMARIES_FORCE_REBUILD else load_json(SUMMARIES_PATH, default=[])
summary_map = {item['cluster_id']: item for item in existing}
all_summaries = list(existing)
if SUMMARIES_FORCE_REBUILD and SUMMARIES_PATH.exists():
    SUMMARIES_PATH.unlink()

label_examples = ', '.join(f'"{x}"' for x in SUMMARY_LABEL_EXAMPLES)
direction_guide = (
    f'Use labels that clearly distinguish favoring {OPTION_A_LABEL}, favoring {OPTION_B_LABEL}, '
    'or balancing both. Avoid generic labels like "Perspective 1" or broad topics like "Financial Decision".'
)

system_prompt = (
    'You summarize structured financial perspectives. Use only the claims, conditions, caveats, and reasons provided. '
    'Do not add outside information. Distinguish the central claim from its conditions. '
    'Create a short, contrastive viewpoint label that sounds like a stance, not a generic topic heading.'
)

for cluster in tqdm(clusters, desc='Summarizing perspectives'):
    if cluster['cluster_id'] in summary_map:
        continue

    claim_block = []
    for idx, item in enumerate(cluster['passages'], start=1):
        claim_block.append(
            f"{idx}. Claim: {item['main_claim']}\n"
            f"Support: {item['supporting_text']}\n"
            f"Conditions: {item['conditions']}\n"
            f"Caveats: {item['caveats']}\n"
            f"Reason types: {item['reason_types']}\n"
            f"Source: {item['source_name']} ({item['source_type']})"
        )

    user_prompt = f'''Query: {QUERY_TEXT}
Option A: {OPTION_A_LABEL}
Option B: {OPTION_B_LABEL}
Perspective cluster: {cluster['cluster_id']}

The following structured arguments belong to one broader perspective.
Summarize the shared main claim first, then mention the common conditions and caveats.
{direction_guide}
Example label styles: {label_examples}

{chr(10).join(claim_block)}

Return ONLY valid JSON:
{{
  "label": "2-8 word contrastive viewpoint label",
  "summary": "3-4 sentence summary of the central perspective",
  "key_reasons": ["reason 1", "reason 2", "reason 3"],
  "conditions": ["condition 1", "condition 2"],
  "caveats": ["caveat 1", "caveat 2"],
  "confidence_label": "Confident" or "Conditional" or "Informational"
}}'''

    result = call_openai_json('gpt-4o', system_prompt, user_prompt, temperature=0.2)
    summary_obj = {
        'cluster_id': cluster['cluster_id'],
        'label': result.get('label', cluster['cluster_id']),
        'summary': result.get('summary', '').strip(),
        'key_reasons': result.get('key_reasons', []),
        'conditions': result.get('conditions', cluster['top_conditions'][:3]),
        'caveats': result.get('caveats', cluster['top_caveats'][:3]),
        'confidence_label': result.get('confidence_label', confidence_label_from_postures(cluster['posture_distribution'])),
        'source_types': list(cluster['source_types']),
        'source_names': list(cluster.get('source_names', [])),
        'source_count': len(cluster.get('source_names', [])),
    }
    all_summaries.append(summary_obj)
    summary_map[cluster['cluster_id']] = summary_obj
    save_json(SUMMARIES_PATH, all_summaries)

save_json(SUMMARIES_PATH, all_summaries)
print(f'Saved {len(all_summaries)} summaries to {SUMMARIES_PATH}')

elapsed = stop_timer('step_8_summarization_v4', step_start)
print(f'Step 8 time: {elapsed:.2f}s')


## STEP 9: Final output

In [ ]:
step_start = start_timer()
clusters = load_json(CLUSTERS_PATH, default=[])
summaries = load_json(SUMMARIES_PATH, default=[])
summary_map = {item['cluster_id']: item for item in summaries}

final_output = []
for idx, cluster in enumerate(sorted(clusters, key=lambda c: c['mean_confidence'], reverse=True), start=1):
    summary = summary_map.get(cluster['cluster_id'], {})
    item = {
        'perspective_id': f'p{idx}',
        'perspective_number': idx,
        'cluster_id': cluster['cluster_id'],
        'label': summary.get('label', cluster['cluster_id']),
        'confidence': summary.get('confidence_label', confidence_label_from_postures(cluster['posture_distribution'])),
        'source_types': summary.get('source_types', cluster['source_types']),
        'source_names': summary.get('source_names', cluster.get('source_names', [])),
        'source_count': summary.get('source_count', len(cluster.get('source_names', []))),
        'summary': summary.get('summary', ''),
        'key_reasons': summary.get('key_reasons', []),
        'conditions': summary.get('conditions', cluster['top_conditions'][:3]),
        'caveats': summary.get('caveats', cluster['top_caveats'][:3]),
    }
    final_output.append(item)

    print(f'=== PERSPECTIVE {idx} ===')
    print(f"Label: {item['label']}")
    print(f"Confidence: {item['confidence']}")
    print(f"Sources: {item['source_types']}")
    print(f"Summary: {item['summary']}")
    print('Key reasons:')
    for reason in item['key_reasons']:
        print(f'  - {reason}')
    print('Conditions:')
    for condition in item['conditions']:
        print(f'  - {condition}')
    print('Caveats:')
    for caveat in item['caveats']:
        print(f'  - {caveat}')
    print('')

save_json(FINAL_OUTPUT_PATH, final_output)

elapsed = stop_timer('step_9_final_output_v4', step_start)
print(f'Step 9 time: {elapsed:.2f}s')


## STEP 10: Main viewpoint selection

This display-only step selects 3 primary viewpoints from the discovered clusters.

CHANGE FROM EPIC v4 core pipeline:
- the core discovery stage may produce more than 3 valid perspectives
- this step chooses 3 for interface prominence using confidence plus distinctness
- it does **not** use raw support count as the dominant signal, which helps minority but well-formed viewpoints survive

In [ ]:
step_start = start_timer()
clusters = load_json(CLUSTERS_PATH, default=[])
summaries = load_json(SUMMARIES_PATH, default=[])
summary_map = {item['cluster_id']: item for item in summaries}

DISPLAY_TOP_N = 3
DISPLAY_SELECTION_PATH = RUN_DIR / 'main_viewpoints.json'
SIMILARITY_REDUNDANCY_THRESHOLD = 0.82

if not clusters:
    raise ValueError('No clusters found. Run Step 6 onward first.')

# CHANGE FROM CORE DISCOVERY:
# We select display viewpoints using confidence plus semantic distinctness, not just cluster size.
# We also explicitly prefer directional coverage first so the interface does not show three near-duplicate
# clusters from the same stance family when a distinct option_b or hybrid perspective exists.

def dominant_direction(cluster):
    dist = cluster.get('coarse_direction_distribution', {})
    if not dist:
        return 'unclear'
    return max(dist.items(), key=lambda kv: kv[1])[0]

centroids = {c['cluster_id']: np.array(c['centroid'], dtype=float) for c in clusters}
conf_values = [float(c.get('mean_confidence', 0.0)) for c in clusters]
size_values = [int(c.get('size', 0)) for c in clusters]
min_conf, max_conf = min(conf_values), max(conf_values)
min_size, max_size = min(size_values), max(size_values)


def normalize(value, min_v, max_v):
    if max_v <= min_v:
        return 1.0
    return float((value - min_v) / (max_v - min_v))


def cosine_sim(cluster_a, cluster_b):
    return float(cosine_similarity([centroids[cluster_a['cluster_id']]], [centroids[cluster_b['cluster_id']]])[0][0])


def min_distance_to_selected(cluster, selected):
    if not selected:
        return 1.0
    sims = [cosine_sim(cluster, other) for other in selected]
    return float(1 - max(sims))


def display_score(cluster, selected):
    conf_score = normalize(float(cluster.get('mean_confidence', 0.0)), min_conf, max_conf)
    size_score = normalize(float(cluster.get('size', 0)), min_size, max_size)
    distance_score = min_distance_to_selected(cluster, selected)
    max_similarity = max((cosine_sim(cluster, other) for other in selected), default=0.0)
    redundancy_penalty = max(0.0, (max_similarity - SIMILARITY_REDUNDANCY_THRESHOLD) / max(1e-6, 1 - SIMILARITY_REDUNDANCY_THRESHOLD))
    return 0.45 * conf_score + 0.40 * distance_score + 0.05 * size_score - 0.20 * redundancy_penalty


available = sorted(clusters, key=lambda c: (c.get('mean_confidence', 0.0), c.get('size', 0)), reverse=True)
selected = []
remaining = list(available)

# 1. Directional coverage first: try to keep one strong viewpoint from each major family.
for direction in ['option_a', 'option_b', 'hybrid']:
    candidates = [c for c in remaining if dominant_direction(c) == direction]
    if not candidates:
        continue
    if not selected:
        best = candidates[0]
    else:
        best = max(candidates, key=lambda c: display_score(c, selected))
    selected.append(best)
    remaining = [c for c in remaining if c['cluster_id'] != best['cluster_id']]
    if len(selected) >= DISPLAY_TOP_N:
        break

# 2. If we still need slots, fill them with the strongest distinct candidates.
while remaining and len(selected) < min(DISPLAY_TOP_N, len(clusters)):
    best = max(remaining, key=lambda c: display_score(c, selected))
    selected.append(best)
    remaining = [c for c in remaining if c['cluster_id'] != best['cluster_id']]

main_viewpoints = []
for rank, cluster in enumerate(selected, start=1):
    summary = summary_map.get(cluster['cluster_id'], {})
    record = {
        'main_viewpoint_rank': rank,
        'cluster_id': cluster['cluster_id'],
        'label': summary.get('label', cluster['cluster_id']),
        'confidence': summary.get('confidence_label', confidence_label_from_postures(cluster['posture_distribution'])),
        'mean_confidence': float(cluster.get('mean_confidence', 0.0)),
        'size': int(cluster.get('size', 0)),
        'dominant_direction': dominant_direction(cluster),
        'source_types': summary.get('source_types', cluster.get('source_types', [])),
        'source_names': summary.get('source_names', cluster.get('source_names', [])),
        'source_count': summary.get('source_count', len(cluster.get('source_names', []))),
        'summary': summary.get('summary', ''),
        'key_reasons': summary.get('key_reasons', []),
        'conditions': summary.get('conditions', cluster.get('top_conditions', [])[:3]),
        'caveats': summary.get('caveats', cluster.get('top_caveats', [])[:3]),
    }
    main_viewpoints.append(record)

save_json(DISPLAY_SELECTION_PATH, main_viewpoints)

print('=== MAIN VIEWPOINTS FOR INTERFACE ===')
for item in main_viewpoints:
    print(f"[{item['main_viewpoint_rank']}] {item['label']} | cluster={item['cluster_id']} | direction={item['dominant_direction']} | confidence={item['confidence']} | size={item['size']} | mean_conf={item['mean_confidence']:.3f}")

elapsed = stop_timer('step_10_main_viewpoint_selection_v4', step_start)
print(f'Step 10 time: {elapsed:.2f}s')
print(f'Saved main viewpoints to {DISPLAY_SELECTION_PATH}')

## Final Summary

In [ ]:
clean_docs = load_json(CLEAN_PATH, default=[])
retrieved_docs = load_json(RETRIEVED_DOCS_PATH, default=[])
arguments = load_json(STRUCTURED_PATH, default=[])
clusters = load_json(CLUSTERS_PATH, default=[])
evaluation = load_json(EVAL_PATH, default={})
timings = load_json(TIMINGS_PATH, default={})

epic_metrics = evaluation.get('EPIC_v4', evaluation.get('EPIC_v3', {}))
semantic_metrics = evaluation.get('SemanticOnlyGlobal', {})
random_metrics = evaluation.get('Random', {})

print(f'Run name: {RUN_NAME}')
print(f'Query: {QUERY_TEXT}')
print(f'Option A: {OPTION_A_LABEL}')
print(f'Option B: {OPTION_B_LABEL}')
print(f'Total documents processed: {len(clean_docs)}')
print(f'Total retrieved candidate docs: {len(retrieved_docs)}')
print(f'Total structured arguments: {len(arguments)}')
print(f'Total perspectives formed: {len(clusters)}')
print(f'EPIC_v4 ECS score: {epic_metrics.get("ecs", 0.0):.2f}')
print(f'EPIC_v4 vs Random ECS improvement: {epic_metrics.get("ecs", 0.0) - random_metrics.get("ecs", 0.0):.2f}')
print(f'EPIC_v4 vs SemanticOnlyGlobal ECS delta: {epic_metrics.get("ecs", 0.0) - semantic_metrics.get("ecs", 0.0):.2f}')
print('Time taken per step:')
for step_name, seconds in timings.items():
    print(f'  {step_name}: {seconds:.2f}s')
